In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install matplotlib
!pip install matplotlib seaborn


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
df = pd.read_csv('encoded_dataset_28_5_25 (1).csv')
df.head()

In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

In [ ]:
numeric_cols = ['AGE', 'WARD_encoded', 'MONTH_sin', 'MONTH_cos']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()  # Preserve original X
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])  # Scale only numeric columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
knn = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [5, 7],        # reduced from 5 values → 2 values
    'weights': ['uniform'],       # keep only one option
    'metric': ['euclidean']       # keep only one distance metric
}


In [ ]:
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

In [ ]:

print("Best parameters from GridSearchCV:", grid_search.best_params_)
print("Best cross-validation F1-score:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
round(accuracy_score(y_test, y_pred)*100,2)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dengue (0)', 'Malaria (1)'], yticklabels=['Dengue (0)', 'Malaria (1)'])
plt.title('Confusion Matrix for K-Nearest Neighbors (Best Model)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('knn_cm_bfrRESamp.png', dpi=400)
plt.show()

# **LIME Explanation**

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=['Dengue (0)', 'Malaria (1)'],
    mode='classification',
    random_state=42
)

In [ ]:
instance_idx = 0
instance = X_test.iloc[instance_idx].values
lime_exp = lime_explainer.explain_instance(
    data_row=instance,
    predict_fn=best_model.predict_proba,
    num_features=len(X.columns)  # Include all features to aggregate later
)

In [ ]:
len(X.columns)

In [ ]:
instance


In [ ]:
len(instance)

In [ ]:
num_instances = 10

print("The 10 instances that WILL be used:")
print(X_test.iloc[:num_instances])

In [ ]:
print("Dataset indices of the 10 instances:")
print(X_test.index[:num_instances].tolist())

In [ ]:
lime_exp.show_in_notebook()

In [ ]:
plt.figure()
lime_exp.as_pyplot_figure()
plt.title('LIME Explanation for First Instance (KNN)')
plt.tight_layout()
plt.savefig('lime_explanation_instance_0_logistic.png', dpi=400)
plt.show()

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data = X_train.values,
    feature_names = X_train.columns.tolist(),
    class_names = ['Dengue(0)', 'Malaria(1)'],   # 0 → Dengue, 1 → Malaria
    mode = 'classification'
)

In [ ]:
num_instances = 10

for i in range(num_instances):
    instance = X_test.iloc[i]

    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # show all  features
    )

    # Show explanation
    explanation.show_in_notebook()

In [ ]:
num_instances = 10

for i in range(num_instances):
    print(f"\nInstance {i}")
    print("Dataset index:", X_test.index[i])
    print(X_test.iloc[i])


In [ ]:
num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    #explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=5)
    explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=len(X.columns))
    a = explanation.as_list()
    # print(explanation.as_list())
    print(a)
    print("\n")

In [ ]:
all_explanations = []

num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # ask for all 22
    )
    all_explanations.append(explanation.as_list())

In [ ]:
from collections import defaultdict
import numpy as np

feature_weights = defaultdict(list)

for exp in all_explanations:
    for feature_cond, weight in exp:
        for col in X.columns:
            if col in feature_cond:
                feature_weights[col].append(abs(weight))
                break

In [ ]:
avg_weight_df = pd.DataFrame({
    "Feature": X.columns,
    "Avg_Absolute_Weight": [
        np.mean(feature_weights[col]) if len(feature_weights[col]) > 0 else 0
        for col in X.columns
    ]
}).sort_values(by="Avg_Absolute_Weight", ascending=False)

print(avg_weight_df)


In [ ]:
len(avg_weight_df)


In [ ]:
thresholds = sorted(
    avg_weight_df["Avg_Absolute_Weight"].unique()
)

print("Threshold values:")
for t in thresholds:
    print(t)



In [ ]:
len(thresholds)

In [ ]:
# Fix ONE sample index
sample_index = 0
x_sample = X_test.iloc[sample_index]

comparison = []

for t in thresholds:
    selected_features = avg_weight_df[
        avg_weight_df["Avg_Absolute_Weight"] >= t
    ]["Feature"].tolist()

    comparison.append({
        "Threshold": t,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df)


In [ ]:
all_samples_comparison = []

num_instances = 10

for i in range(num_instances):
    sample_row = {
        "Sample_Index": i,
        "Threshold_Comparison": []
    }

    for t in thresholds:
        selected_features = avg_weight_df[
            avg_weight_df["Avg_Absolute_Weight"] >= t
        ]["Feature"].tolist()

        sample_row["Threshold_Comparison"].append({
            "Threshold": t,
            "Num_Selected_Features": len(selected_features),
            "Selected_Features": selected_features
        })

    all_samples_comparison.append(sample_row)


In [ ]:
for sample in all_samples_comparison:
    print(f"\n================ Sample {sample['Sample_Index']} ================")

    for item in sample["Threshold_Comparison"]:
        print(f"Threshold: {item['Threshold']}")
        print(f"Num Selected Features: {item['Num_Selected_Features']}")
        print(f"Selected Features: {item['Selected_Features']}")
        print("-" * 60)

In [ ]:
rows = []

for sample in all_samples_comparison:
    for item in sample["Threshold_Comparison"]:
        rows.append({
            "Sample_Index": sample["Sample_Index"],
            "Threshold": item["Threshold"],
            "Num_Selected_Features": item["Num_Selected_Features"],
            "Selected_Features": item["Selected_Features"]
        })

comparison_df = pd.DataFrame(rows)
print(comparison_df)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
results = []

In [ ]:
for _, row in comparison_df.iterrows():

    sample_index = row["Sample_Index"]
    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print(f"Sample {sample_index}")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)

    # Safety check
    if len(selected_features) == 0:
        print("No features selected. Skipping.")
        continue

In [ ]:
import pandas as pd

# If comparison_df already exists, this will skip creation
try:
    comparison_df
except NameError:
    print("comparison_df not found. Creating a sample one for testing...")

    comparison_df = pd.DataFrame({
        "Sample_Index": [0],
        "Threshold": [0.5],
        "Selected_Features": [list(X_train.columns[:5])]
    })


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------------------------------
# 1. Take Sample 0 results only
# -------------------------------
sample0_df = comparison_df[comparison_df["Sample_Index"] == 0]

final_results_knn = []

# -------------------------------
# 2. Loop over each threshold
# -------------------------------
for _, row in sample0_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n========================================")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # -------------------------------
    # 3. Create reduced datasets
    # -------------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # -------------------------------
    # 4. Stratified K-Fold
    # -------------------------------
    skf_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # -------------------------------
    # 5. KNN + REDUCED GRID
    # -------------------------------
    knn_model = KNeighborsClassifier()

    param_grid = {
        'n_neighbors': [5, 7],
        'weights': ['uniform'],
        'metric': ['euclidean']
    }

    grid_search_knn = GridSearchCV(
        estimator=knn_model,
        param_grid=param_grid,
        cv=skf_knn,
        scoring='f1',
        n_jobs=-1
    )

    # Train
    grid_search_knn.fit(X_train_selected, y_train)

    print("Best Parameters:", grid_search_knn.best_params_)
    print("Best CV F1-score:", grid_search_knn.best_score_)

    # -------------------------------
    # 6. Best model
    # -------------------------------
    best_knn_model = grid_search_knn.best_estimator_
    best_knn_model.fit(X_train_selected, y_train)

    # -------------------------------
    # 7. Test on FIRST 10 rows
    # -------------------------------
    X_test_subset = X_test_selected.iloc[:10]
    y_test_subset = y_test.iloc[:10]

    y_pred_subset = best_knn_model.predict(X_test_subset)

    acc_subset = round(accuracy_score(y_test_subset, y_pred_subset) * 100, 2)
    print("Subset Accuracy (%):", acc_subset)

    # -------------------------------
    # 8. Classification Report
    # -------------------------------
    print("\nClassification Report (Subset):")
    print(classification_report(
        y_test_subset,
        y_pred_subset,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # -------------------------------
    # 9. Confusion Matrix
    # -------------------------------
    cm = confusion_matrix(y_test_subset, y_pred_subset)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )

    plt.title(f'KNN Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # 10. Store results
    # -------------------------------
    final_results_knn.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Subset_Accuracy": acc_subset,
        "CV_F1": grid_search_knn.best_score_,
        "Selected_Features": selected_features
    })

# -------------------------------
# 11. Final Results Table
# -------------------------------
final_results_knn_df = pd.DataFrame(final_results_knn)

print("\nFINAL KNN RESULTS:")
print(final_results_knn_df)


# **SHAP Explanations**

In [ ]:
!pip install shap

In [ ]:
import shap

background = shap.sample(X_train, 50)
explainer = shap.KernelExplainer(best_model.predict_proba, background, link="logit")

In [ ]:
X_test_subset_shap = X_test.iloc[:50]

In [ ]:
shap_values = explainer.shap_values(X_test_subset_shap)

In [ ]:
print("Shape of X_test_subset:", X_test_subset_shap.shape)
print("Shape of shap_values_positive:", shap_values.shape)
print("Feature names:", X_test_subset_shap.columns.tolist())

In [ ]:
feature_names = X_test_subset_shap.columns.tolist()

In [ ]:
shap_values = explainer.shap_values(X_test_subset_shap)

In [ ]:
shap_values_positive = shap_values[..., 1]

In [ ]:
shap_values_positive


In [ ]:
print("Shape of X_test_subset:", X_test_subset_shap.shape)
print("Shape of shap_values_positive:", shap_values_positive.shape)
print("Feature names:", X_test_subset_shap.columns.tolist())

In [ ]:
feature_names = X_test_subset_shap.columns.tolist()

In [ ]:
#Beeswarm Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_positive, X_test_subset_shap, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm Summary Plot for Positive Class")
plt.tight_layout()
plt.savefig('shap_beeswarm_plot_KNN_before_resampling.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
#Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_positive, X_test_subset_shap, feature_names=feature_names, plot_type="bar", show=False)
plt.title("SHAP Bar Plot for Positive Class")
plt.tight_layout()
plt.savefig('shap_bar_plot_KNN_before_resampling.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
shap_values_negative = shap_values[..., 0]

In [ ]:
shap_values_negative

In [ ]:
import numpy as np

if isinstance(shap_values_positive, shap.Explanation):
    mean_abs_shap = np.abs(shap_values_positive.values).mean(axis=0)
else:
    mean_abs_shap = np.abs(shap_values_positive).mean(axis=0)

shap_importance = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Abs_SHAP': mean_abs_shap
})

In [ ]:
shap_importance = shap_importance.sort_values(by='Mean_Abs_SHAP', ascending=False)

In [ ]:
n_features = len(feature_names)
n_features

In [ ]:
import numpy as np
import pandas as pd

# Mean absolute SHAP for both classes
mean_abs_shap_class_0 = np.mean(np.abs(shap_values_negative), axis=0)
mean_abs_shap_class_1 = np.mean(np.abs(shap_values_positive), axis=0)

# Global importance
mean_abs_shap_global = (mean_abs_shap_class_0 + mean_abs_shap_class_1) / 2


In [ ]:
feature_names = X_train.columns.tolist()

# Unique sorted thresholds
shap_thresholds = sorted(np.unique(mean_abs_shap_global))

shap_results = []

for t in shap_thresholds:

    mask = mean_abs_shap_global >= t
    selected_features = [feature_names[i] for i in range(len(feature_names)) if mask[i]]

    if len(selected_features) == 0:
        continue

    shap_results.append({
        "Threshold": float(t),
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

# Create dataframe
shap_feature_selection_df = pd.DataFrame(shap_results)

print(shap_feature_selection_df.head())


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

final_results_shap_knn = []

for _, row in shap_feature_selection_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print("Threshold:", threshold)
    print("Number of Features:", len(selected_features))

    # Reduce datasets
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # GridSearch
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    knn = KNeighborsClassifier()

    param_grid = {
        'n_neighbors': [5, 7],
        'weights': ['uniform'],
        'metric': ['euclidean']
    }

    grid = GridSearchCV(knn, param_grid, cv=skf, scoring='f1', n_jobs=-1)
    grid.fit(X_train_selected, y_train)

    best_model = grid.best_estimator_

    # Test subset
    X_test_subset = X_test_selected.iloc[:50]
    y_test_subset = y_test.iloc[:50]

    y_pred = best_model.predict(X_test_subset)

    acc = accuracy_score(y_test_subset, y_pred)
    f1  = f1_score(y_test_subset, y_pred)

    print("Accuracy:", round(acc*100,2))
    print("F1:", round(f1,4))

    # Confusion matrix
    cm = confusion_matrix(y_test_subset, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Dengue (0)','Malaria (1)'],
                yticklabels=['Dengue (0)','Malaria (1)'])
    plt.title(f'KNN SHAP Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    final_results_shap_knn.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Accuracy": acc,
        "F1_Score": f1
    })
